# RAG-LIBRO — Exploración del pipeline

**Fases cubiertas:** 1a (ingestión) → 1b (FAISS) → 1c (retriever + LCEL) → **1d (`app/rag.py`)**.

Lógica de producto en `app/rag.py`; piezas en `ingest.py`, `vectorstore.py`, `llm.py`. Kernel: `.venv` de `backend/`.

In [ ]:
import sys
from pathlib import Path

BACKEND = Path.cwd()
if not (BACKEND / "app" / "ingest.py").is_file():
    BACKEND = BACKEND.parent  # si cwd es notebooks/
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

from app.ingest import (
    CHUNK_OVERLAP,
    CHUNK_SIZE,
    chunk_length_stats,
    chunks_on_pages,
    get_paths,
    ingest_pdf,
    page_pdf_1based,
    pdf_page_count,
)

## 1.1 — Scaffold: paths, `.env`, check PDF

In [ ]:
paths = get_paths()
print(f"project_root: {paths.project_root}")
print(f"pdf_path:     {paths.pdf_path}")
print(f"storage_dir:  {paths.storage_dir}")
print(f"PDF existe:   {paths.pdf_path.is_file()}")

n_pages = pdf_page_count(paths.pdf_path)
assert paths.pdf_path.is_file(), f"Colocá el PDF en {paths.pdf_path}"
print(f"Páginas PDF:  {n_pages}")

## 1.2 — `PyPDFLoader`: un `Document` por página

`metadata['page']` = 0-based (LangChain). `page_pdf` = 1-based (eval / citas).

In [ ]:
page_docs, chunk_docs = ingest_pdf(paths.pdf_path)
assert len(page_docs) == n_pages, f"{len(page_docs)} docs vs {n_pages} páginas"
print(f"Documents cargados: {len(page_docs)}")

for i in (0, 34, 179):  # primera, ~Q01, ~Q05 (0-based page 179 = PDF p.180)
    d = page_docs[i]
    print("---")
    print(f"page (0-based): {d.metadata.get('page')} | page_pdf: {page_pdf_1based(d)}")
    print(d.page_content[:400].replace("\n", " ") + "...")

## 1.3 — `RecursiveCharacterTextSplitter` (1000 / 200)

In [ ]:
stats = chunk_length_stats(chunk_docs)
print(f"chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")
print(f"Total chunks: {stats['count']}")
print(f"Longitud (chars): min={stats['min']}, max={stats['max']}, mean={stats['mean']}")

q05 = chunks_on_pages(chunk_docs, {180, 181})
print(f"\nGate Q05 — chunks en p.180-181: {len(q05)}")
if q05:
    s = q05[0]
    print(f"  page_pdf={page_pdf_1based(s)}, len={len(s.page_content)}")
    print(f"  snippet: {s.page_content[:300].replace(chr(10), ' ')}...")

### Gate Fase 1a

- [x] PDF ~542 páginas
- [x] `len(page_docs) == n_pages`
- [x] Chunks con `page_pdf` en metadata
- [x] Al menos un chunk en p.180–181 (Q05)

**Siguiente:** Fase 1b — embeddings MiniLM + FAISS persistido.

## Fase 1b — FAISS index (bridge)

Lógica en `app/vectorstore.py`. Carga idempotente: si `storage/faiss_index/` existe, reutiliza.
Si no existe (o `REBUILD_INDEX=true`), embeddea y persiste.

In [ ]:
from app.rag import build_or_load_vectorstore
from app.vectorstore import create_embeddings, default_index_dir, index_files_exist

embeddings = create_embeddings()
vectorstore = build_or_load_vectorstore(chunk_docs, embeddings=embeddings)

print(f"Index dir:  {default_index_dir()}")
print(f"Index files: {index_files_exist(default_index_dir())}")
print(f"Vectorstore type: {type(vectorstore).__name__}")

## Fase 1c — Retriever y LCEL chain

Conectamos el índice FAISS → retriever → prompt → LLM → respuesta parseada.

### 1.7 — Retriever `k=4` + smoke test Q01 / Q05

In [ ]:
from app.ingest import page_pdf_1based
from app.eval_cases import EVAL_CASES
from app.rag import RETRIEVER_K_DEFAULT, get_retriever

RETRIEVER_K = RETRIEVER_K_DEFAULT
retriever = get_retriever(k=RETRIEVER_K, vectorstore=vectorstore)

def smoke_retrieval(query: str, expected_pages: tuple[int, ...]):
    """Recupera k docs y muestra si las páginas esperadas están en top-k."""
    docs = retriever.invoke(query)
    retrieved_pages = [page_pdf_1based(d) for d in docs]
    hit = any(p in retrieved_pages for p in expected_pages)
    print(f"Query: {query[:60]}...")
    print(f"  Expected pages: {expected_pages}")
    print(f"  Retrieved pages: {retrieved_pages}")
    print(f"  HIT: {'✓' if hit else '✗'}")
    print()
    return docs

q01 = next(c for c in EVAL_CASES if c.id == "Q01")
q05 = next(c for c in EVAL_CASES if c.id == "Q05")

docs_q01 = smoke_retrieval(q01.query, q01.expected_pages)
docs_q05 = smoke_retrieval(q05.query, q05.expected_pages)

### 1.8 — `ChatPromptTemplate` + `format_docs`

Instrucciones al LLM:
- Responder **solo** con el contexto provisto.
- Citar las páginas de donde saca la información.
- Decir "No tengo información suficiente" si el contexto no alcanza.

In [ ]:
from app.rag import RAG_PROMPT, format_docs

prompt = RAG_PROMPT

# Verificar renderizado del prompt con Q05
sample_context = format_docs(docs_q05)
rendered = prompt.format(context=sample_context, question=q05.query)
print(rendered[:800])
print(f"\n... ({len(rendered)} chars total)")

### 1.9 — Chain LCEL: retriever → prompt → LLM → parser

Operador `|` (pipe) compone Runnables. `RunnablePassthrough` pasa la query,
el retriever busca docs, `format_docs` los convierte a texto, y todo entra al prompt.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from app.llm import get_llm

llm = get_llm()

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Test con Q05
answer_q05 = rag_chain.invoke(q05.query)
print("=" * 60)
print(f"Q05: {q05.query}")
print("=" * 60)
print(answer_q05)

In [ ]:
from app.rag import answer_with_sources

# Gate 1.11–1.12: mismo contrato que test_eval.py (import desde app.rag)
result_q01 = answer_with_sources(q01.query, k=RETRIEVER_K)
print(f"Q01 pages: {result_q01['pages']}")
print(f"Q01 answer: {result_q01['answer'][:200]}...")
print()

result_q05 = answer_with_sources(q05.query, k=RETRIEVER_K)
print(f"Q05 pages: {result_q05['pages']}")
print(f"Q05 answer: {result_q05['answer'][:200]}...")

### 1.10 — Verificar `get_llm(provider=...)` desde notebook

Probamos que la cadena funciona con cada provider explícito (mock como mínimo garantizado).
El test automatizado vive en `tests/test_llm_fallback.py`.

In [ ]:
from app.llm import get_llm, Provider
from langchain_core.messages import HumanMessage

providers_to_test: list[Provider] = ["mock"]

# Agregar providers reales si tienen key configurada
import os
if os.getenv("GROQ_API_KEY", "").strip():
    providers_to_test.append("groq")
if os.getenv("OPENROUTER_API_KEY", "").strip():
    providers_to_test.append("openrouter")

print(f"Providers disponibles: {providers_to_test}\n")

test_query = "What is RAG according to this book?"
sample_context = format_docs(docs_q05[:2])

for provider in providers_to_test:
    print(f"--- Provider: {provider} ---")
    try:
        test_llm = get_llm(provider=provider)
        test_chain = (
            {"context": lambda _: sample_context, "question": RunnablePassthrough()}
            | prompt
            | test_llm
            | StrOutputParser()
        )
        out = test_chain.invoke(test_query)
        print(f"  ✓ Respuesta ({len(out)} chars): {out[:150]}...")
    except Exception as e:
        print(f"  ✗ Error: {type(e).__name__}: {e}")
    print()

### Gate Fase 1c + 1d

- [ ] Retriever `k=4` devuelve docs con `page_pdf` 1-based
- [ ] Q05: p.180–181 presente en top-4
- [ ] Prompt renderizado incluye contexto + regla "no sé"
- [ ] `answer_with_sources` desde `app.rag` devuelve `answer` + `pages`
- [ ] `get_llm(provider="mock")` funciona en la chain
- [ ] `pytest tests/test_llm_fallback.py` y `pytest tests/test_rag.py` verde

**Siguiente:** Fase 1e — `run_eval_suite` y tuning (≥70% PASS).

## Fase 1e — Evaluación y tuning

**Objetivo:** medir el pass rate real con el pipeline completo y ajustar parámetros hasta ≥ 70 % PASS.

Pipeline de eval:
1. `run_eval_suite` corre las 10 queries del `EVAL.md` contra el RAG real.
2. Para cada query se chequean **dos criterios independientes**:
   - **Criterio A (retrieval):** ¿alguna de las páginas esperadas aparece en el top-k recuperado por FAISS?
   - **Criterio B (generation):** ¿la respuesta del LLM contiene todos los conceptos clave?
3. Una query pasa solo si ambos criterios son PASS.

El diagnóstico A/B orienta el tuning: si falla A → tocar FAISS/chunking; si falla B → tocar el prompt o el LLM.

### 1.13 — `run_eval_suite`: tabla A/B por query (baseline k=4)

In [ ]:
from tests.test_eval import run_eval_suite
from app.rag import answer_with_sources

def make_runner(k: int):
    """Crea una función que llama answer_with_sources con un k fijo."""
    def run(query: str, **_kw):
        result = answer_with_sources(query, k=k)
        return result["answer"], result["pages"]
    return run

def print_ab_table(report: dict, k: int) -> None:
    """Imprime tabla con resultado A (retrieval) y B (generation) por query."""
    header = f"{'ID':<5} {'A-Retrieval':<13} {'B-Generation':<14} {'Global':<8}  Páginas recuperadas"
    sep = "-" * 70
    print(f"\n{'='*70}")
    print(f"  EVAL SUITE  k={k}")
    print(f"{'='*70}")
    print(header)
    print(sep)
    for r in report["results"]:
        a = "✓ PASS" if r["retrieval_ok"] else "✗ FAIL"
        b = "✓ PASS" if r["answer_ok"] else "✗ FAIL"
        g = "✓" if r["passed"] else "✗"
        print(f"{r['id']:<5} {a:<13} {b:<14} {g:<8}  {r['retrieved_pages']}")
    print(sep)
    pct = report["pass_rate"] * 100
    baseline = "✓ BASELINE ALCANZADO (≥70%)" if report["meets_baseline"] else "✗ Bajo baseline (<70%)"
    print(f"  PASS: {report['passed']}/{report['total']} ({pct:.0f}%)  →  {baseline}")
    print(f"{'='*70}\n")

# Baseline: k=4 (parámetros iniciales de Fase 1)
K_BASELINE = 4
report_k4 = run_eval_suite(make_runner(K_BASELINE), k=K_BASELINE)
print_ab_table(report_k4, K_BASELINE)

### 1.15 — Diagnóstico: ¿dónde fallan las queries?

Regla de diagnóstico:
- **Solo A falla** → el retriever no está encontrando los chunks correctos → tocar `k`, `chunk_size`, `overlap`
- **Solo B falla** → el retriever funciona pero el LLM no usa bien el contexto → tocar el prompt o el modelo
- **Ambos fallan** → el retriever no encuentra nada útil; arreglar retrieval primero

In [ ]:
def diagnose(report: dict, k: int) -> dict:
    """Clasifica cada FAIL en: solo-A, solo-B, ambos. Devuelve dict con listas."""
    results = report["results"]
    only_a_fail = [r["id"] for r in results if not r["retrieval_ok"] and r["answer_ok"]]
    only_b_fail = [r["id"] for r in results if r["retrieval_ok"] and not r["answer_ok"]]
    both_fail   = [r["id"] for r in results if not r["retrieval_ok"] and not r["answer_ok"]]
    passed      = [r["id"] for r in results if r["passed"]]

    print(f"Diagnóstico k={k}:")
    print(f"  PASS:              {passed}")
    print(f"  FAIL solo A (ret): {only_a_fail}  → tuning retrieval")
    print(f"  FAIL solo B (gen): {only_b_fail}  → tuning generación")
    print(f"  FAIL ambos A+B:    {both_fail}   → retrieval primero")

    if only_a_fail or both_fail:
        print("\n  Acción: subir k o ajustar chunk_size/overlap")
    if only_b_fail:
        print("\n  Acción: prompt más estricto o modelo con más contexto")

    return {"only_a": only_a_fail, "only_b": only_b_fail, "both": both_fail, "passed": passed}

diag_k4 = diagnose(report_k4, K_BASELINE)

### 1.16 — Tuning de retrieval: sweep k=4→6→8

Probamos **una variable a la vez**: solo `k` cambia. Si mejora el pass rate, es señal de que el retriever
necesita más documentos para encontrar las páginas esperadas (casos cross-chapter son los más sensibles).

Por qué una variable a la vez: si cambiáramos `k` y `chunk_size` al mismo tiempo y el resultado mejorara,
no sabríamos a cuál de los dos atribuirle la mejora. Control experimental básico.

In [ ]:
k_reports = {"k4": report_k4}

for k_val in [6, 8]:
    r = run_eval_suite(make_runner(k_val), k=k_val)
    k_reports[f"k{k_val}"] = r
    print_ab_table(r, k_val)
    diagnose(r, k_val)
    print()

# Comparativa de pass_rate
print("\n=== Comparativa sweep k ===")
for label, rep in k_reports.items():
    bar = "█" * rep["passed"] + "░" * (10 - rep["passed"])
    print(f"  {label}:  {bar}  {rep['passed']}/10  ({rep['pass_rate']*100:.0f}%)")

best_k = max(k_reports, key=lambda kk: k_reports[kk]["pass_rate"])
print(f"\nMejor k: {best_k} ({k_reports[best_k]['pass_rate']*100:.0f}% PASS)")

### 1.17 — Tuning de generación (si A está ok, B falla)

Si el retrieval ya funciona (Criterio A = PASS) pero el LLM no usa bien el contexto (Criterio B = FAIL),
las palancas son:

- **Prompt más estricto**: reforzar la regla "respondé solo con el contexto, sin inventar"
- **`temperature` más baja**: el LLM se vuelve más conservador y menos propenso a alucinaciones (inventar información que no está en el contexto)
- **Modelo real vs mock**: el mock (usado en CI) devuelve texto genérico; los proveedores reales (Groq, OpenRouter) generan respuestas usando el contexto recuperado

**Nota**: solo tiene sentido tocar la generación si A ya pasa. Si A falla, cualquier cambio en el LLM es inútil porque el contexto que recibe ya está mal.

In [ ]:
import os
from app.eval_cases import EVAL_CASES
from app.eval_checks import retrieval_passes, answer_passes
from app.rag import get_retriever, format_docs, RAG_PROMPT
from app.llm import get_llm
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Prompt estricto (1.17): agrega énfasis en no alucinar
STRICT_PROMPT_EXTRA = """
IMPORTANT: Do NOT add information that is not explicitly present in the context above.
If you are not certain, say "No tengo información suficiente en el contexto proporcionado."
"""

def answer_strict(query: str, k: int) -> tuple[str, list[int]]:
    """Respuesta con prompt más estricto (anti-alucinación)."""
    from langchain_core.prompts import ChatPromptTemplate
    from app.ingest import page_pdf_1based

    strict_system = (
        "You are a helpful assistant answering questions about the book "
        '"30 Agents Every AI Engineer Must Build" by Imran Ahmad.\n\n'
        "Rules:\n"
        "1. Answer ONLY based on the provided context excerpts.\n"
        "2. Cite the page numbers (e.g., [p.34, p.35]) from which you derive your answer.\n"
        "3. If the context does not contain enough information, say: "
        '"No tengo información suficiente en el contexto proporcionado."\n'
        "4. Be concise but precise.\n"
        "5. Do NOT add any information not present in the context. No guesses, no general knowledge."
    )
    strict_prompt = ChatPromptTemplate.from_messages([
        ("system", strict_system),
        ("human", "Context:\n{context}\n\nQuestion: {question}"),
    ])

    retriever = get_retriever(k=k)
    docs = retriever.invoke(query)
    from app.ingest import page_pdf_1based as p1b
    pages = sorted({p for d in docs if (p := p1b(d))})
    chain = (
        {"context": RunnableLambda(lambda _: format_docs(docs)), "question": RunnablePassthrough()}
        | strict_prompt
        | get_llm()
        | StrOutputParser()
    )
    return chain.invoke(query), pages

# Usar el mejor k del sweep
BEST_K = int(best_k.replace("k", ""))
print(f"Corriendo tuning generación con k={BEST_K} + prompt estricto...\n")

results_strict = []
for case in EVAL_CASES:
    answer, pages = answer_strict(case.query, BEST_K)
    ret_ok = retrieval_passes(pages, list(case.expected_pages))
    ans_ok = answer_passes(answer, list(case.key_concepts))
    results_strict.append({
        "id": case.id,
        "retrieval_ok": ret_ok,
        "answer_ok": ans_ok,
        "passed": ret_ok and ans_ok,
        "retrieved_pages": pages,
    })

passed_strict = sum(1 for r in results_strict if r["passed"])
rate_strict = passed_strict / len(results_strict)
report_strict = {
    "results": results_strict,
    "passed": passed_strict,
    "total": len(results_strict),
    "pass_rate": rate_strict,
    "meets_baseline": rate_strict >= 0.70,
}
print_ab_table(report_strict, BEST_K)
print(f"(vs baseline k=4: {report_k4['passed']}/10 → strict k={BEST_K}: {passed_strict}/10)")

### Gate Fase 1-e

- [ ] `pass_rate >= 0.70` (≥ 7/10 queries PASS)
- [ ] Diagnóstico A/B documentado (patrón de fallas identificado)
- [ ] Tuning registrado: qué variable se tocó y qué delta produjo
- [ ] `EVAL.md` actualizado con corrida real
- [ ] ADR-06 en `PROJECT_OVERVIEW.md`

**Siguiente:** Fase 2 — FastAPI + SSE (conectar `app/rag.py` a endpoints HTTP)